## Notebook to explore synthetic generation of PSDs

In [48]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import sys
import pandas as pd 
import json 
import os
from pathlib import Path
from carrier_generator import CarrierGenerator, CarrierConfig
from psd_simulator import PSDSimulator

from satellite_downlink_simulator.simulation import SpectrumRecord

from satellite_downlink_simulator import (
    Carrier, Transponder, Beam,
    CarrierType, ModulationType, CarrierStandard,
    Band, Polarization, BeamDirection,
    generate_psd, generate_iq,
)

In [53]:
# Pattern of Life simulations
# Controlled example first 
carrier_gen = CarrierGenerator(seed=42)
carrier_config = carrier_gen.generate_carriers(
    num_static_per_xpdr=(0, 1),  # Fewer carriers for quick test
    num_dynamic=5
)

simulator = PSDSimulator(
    carrier_config=carrier_config,
    interferer_configs=[],
    rbw_hz=100e3,  # 100 kHz RBW for faster generation
    vbw_hz=1e3
)

simulator.run_simulation(export_json=True)

Generating static carriers...
  Created 3 static carriers
Generating 5 dynamic carriers...
  Created 5 dynamic carriers
Total carriers: 8
Initialized simulator with 6 transponders
  RBW: 100.0 kHz, VBW: 1.0 kHz

Running 24.0-hour simulation...
  Snapshot interval: 5.0 min
  Total snapshots: 289
  Processing t=0.0 hrs (1/289 snapshots, 0%)
  Processing t=1.0 hrs (13/289 snapshots, 4%)
  Processing t=2.0 hrs (25/289 snapshots, 9%)
  Processing t=3.0 hrs (37/289 snapshots, 13%)
  Processing t=4.0 hrs (49/289 snapshots, 17%)
  Processing t=5.0 hrs (61/289 snapshots, 21%)
  Processing t=6.0 hrs (73/289 snapshots, 25%)
  Processing t=7.0 hrs (85/289 snapshots, 29%)
  Processing t=8.0 hrs (97/289 snapshots, 34%)
  Processing t=9.0 hrs (109/289 snapshots, 38%)
  Processing t=10.0 hrs (121/289 snapshots, 42%)
  Processing t=11.0 hrs (133/289 snapshots, 46%)
  Processing t=12.0 hrs (145/289 snapshots, 50%)
  Processing t=13.0 hrs (157/289 snapshots, 54%)
  Processing t=14.0 hrs (169/289 snapshot

([PSDSnapshot(time_min=np.float64(0.0), frequency_hz=array([1.22000e+10, 1.22001e+10, 1.22002e+10, ..., 1.24158e+10,
         1.24159e+10, 1.24160e+10], shape=(2166,)), psd_dbm_hz=array([-123.06860126, -122.89498251, -122.65840137, ..., -122.70853072,
         -122.85876438, -123.01186889], shape=(2166,)), active_carriers=['Static_1_0', 'Static_4_0', 'Static_5_0'], active_interferers=[], num_carriers=3, num_interferers=0, interferer_frequencies_hz=[], interferer_bandwidths_hz=[]),
  PSDSnapshot(time_min=np.float64(5.0), frequency_hz=array([1.22000e+10, 1.22001e+10, 1.22002e+10, ..., 1.24158e+10,
         1.24159e+10, 1.24160e+10], shape=(2166,)), psd_dbm_hz=array([-123.03744686, -122.89553458, -122.73747769, ..., -122.6715314 ,
         -122.88144496, -123.14735418], shape=(2166,)), active_carriers=['Static_1_0', 'Static_4_0', 'Static_5_0'], active_interferers=[], num_carriers=3, num_interferers=0, interferer_frequencies_hz=[], interferer_bandwidths_hz=[]),
  PSDSnapshot(time_min=np.fl

In [ ]:
# Load to dataframe 
file_path = 'output/spectrum_records_20251020-000157.json'
records = SpectrumRecord.from_file(str(file_path))

In [55]:
# Convert records to DataFrame
data_rows = []
for record in records:
    freq_start = record.cf_hz - (record.bw_hz / 2)
    freq_end = record.cf_hz + (record.bw_hz / 2)
    frequencies = np.linspace(freq_start, freq_end, record.psd_shape[0])
    powers = record.get_psd()
    
    data_rows.append({
        'timestamp': record.timestamp,
        'cf_hz': record.cf_hz,
        'bw_hz': record.bw_hz,
        'rbw_hz': record.rbw_hz,
        'vbw_hz': record.vbw_hz,
        'frequencies': frequencies,
        'powers': powers
    })

df = pd.DataFrame(data_rows)
df = df.set_index('timestamp')

# Verify
print(f"len(df) = {len(df)}, len(records) = {len(records)}")
print(f"df.columns = {list(df.columns)}")
df.head()

len(df) = 289, len(records) = 289
df.columns = ['cf_hz', 'bw_hz', 'rbw_hz', 'vbw_hz', 'frequencies', 'powers']


,cf_hz,bw_hz,rbw_hz,vbw_hz,frequencies,powers
timestamp,,,,,,
2025-10-20 00:01:57.259212,1.230800e+10,216000000.0,100000.0,1000.0,"[12200000000.0, 12200099769.053118, 1220019953...","[-123.06860126194496, -122.89498251216364, -12..."
2025-10-20 00:06:57.259212,1.230800e+10,216000000.0,100000.0,1000.0,"[12200000000.0, 12200099769.053118, 1220019953...","[-123.03744686305575, -122.89553458295725, -12..."
2025-10-20 00:11:57.259212,1.230800e+10,216000000.0,100000.0,1000.0,"[12200000000.0, 12200099769.053118, 1220019953...","[-122.9783909565935, -122.9053796124451, -122...."
2025-10-20 00:16:57.259212,1.230800e+10,216000000.0,100000.0,1000.0,"[12200000000.0, 12200099769.053118, 1220019953...","[-122.98268213082241, -122.72159920834923, -12..."
2025-10-20 00:21:57.259212,1.230800e+10,216000000.0,100000.0,1000.0,"[12200000000.0, 12200099769.053118, 1220019953...","[-123.03803492803544, -122.79180886241463, -12..."


In [57]:
# Option 1: Cluster timestamps based on PSD similarity using 2D Euclidean distance
# Each timestamp is clustered based on its full spectrum pattern
from sklearn.cluster import DBSCAN

# Stack all PSD measurements into a 2D array (timestamps x frequency_bins)
# Each row is a timestamp's complete PSD
psd_matrix = np.vstack(df['powers'].values)
print(f"PSD matrix shape: {psd_matrix.shape} (timestamps x frequency_bins)")

# Run DBSCAN on timestamps with Euclidean distance metric
# eps: maximum Euclidean distance between PSDs to be considered neighbors
# min_samples: minimum number of similar timestamps to form a cluster
# metric: 'euclidean' for 2D Euclidean distance
dbscan_time = DBSCAN(eps=0.5, min_samples=5, metric='euclidean')
time_labels = dbscan_time.fit_predict(psd_matrix)

# Add cluster labels to dataframe
df['time_cluster'] = time_labels

print(f"\nOption 1 - Timestamp Clustering (Euclidean distance):")
print(f"Number of clusters found: {len(set(time_labels)) - (1 if -1 in time_labels else 0)}")
print(f"Number of noise points: {list(time_labels).count(-1)}")
print(f"Cluster distribution:\n{pd.Series(time_labels).value_counts().sort_index()}")

PSD matrix shape: (289, 2166) (timestamps x frequency_bins)

Option 1 - Timestamp Clustering (Euclidean distance):
Number of clusters found: 0
Number of noise points: 289
Cluster distribution:
-1    289
Name: count, dtype: int64


In [56]:
# Visualize records as a waterfall plot
from visualization import Visualizer

# Extract arrays for visualization
time_array = np.array([i * 5.0 for i in range(len(records))])  # Time in minutes (5 min intervals)
frequency_array = df['frequencies'].iloc[0]  # Frequency array (same for all records)
psd_array = np.vstack(df['powers'].values)  # Stack all PSD measurements

# Create a simple metadata object
class SimpleMetadata:
    pass

metadata = SimpleMetadata()

# Create visualizer
viz = Visualizer(
    time_array=time_array,
    frequency_array=frequency_array,
    psd_array=psd_array,
    metadata=metadata,
    output_dir='output/plots/test1'
)

# Create waterfall plot
viz.create_waterfall_plot()

Initialized visualizer
  Time range: 0.0 - 24.0 hrs
  Frequency range: 12.200 - 12.416 GHz
  PSD shape: (289, 2166)

Creating waterfall plot...
  PSD range: -121.0 to -98.1 dBm/Hz
  Saved to output/plots/test1/waterfall_plot.png
